# 14.17 使用時沒有正確答案，怎麼知道該停？


剛才我們畫代價曲線，是因為評估題有正確答案。真正來了一道新題，服務拿不到正確答案，就不能現算每輪 CE，再挑最低的一輪。那樣的停止規則偷偷使用了評估答案。

最簡單的合法做法，是在驗證資料上選一個固定輪數並凍結。另一個做法是讓使用者選[思考投入](https://birdhackor.github.io/tiny-perceptron-vlm/7.23.html)的 low／medium／high 檔位，各對應一個已驗過的輪數。兩者都不需要每題知道何時最好，但平均上可能多算或少算；它們應保留為 Auto（服務先根據輸入替每題選預算，[7.25](https://birdhackor.github.io/tiny-perceptron-vlm/7.25.html)）的比較對照。

要逐題選，可以在開始前讀問題特徵，預測預算；也可以每輪檢查可觀測訊號，例如最大預測機率、前後答案是否一致、狀態改變量。這些是**代理訊號**：能觀察到它們，不等於知道答案對。連續兩輪選同一類，只說明答案穩定了，不說明答對；模型錯得很肯定時也會這樣。

下面是人工機率序列。兩輪 argmax 都是類別0，但正確答案是類別1；它只供本例離線核對，部署時的停止規則拿不到。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

predictions = [torch.tensor([0.95, 0.05]), torch.tensor([0.97, 0.03])]
for r, p in enumerate(predictions, start=1):
    print(r, "選類別", int(p.argmax()), "最大機率", round(float(p.max()), 4))
print("離線核對用的正確答案類別", 1)

`enumerate(predictions, start=1)` 和[1.1](https://birdhackor.github.io/tiny-perceptron-vlm/1.1.html)的 `enumerate` 一樣逐項提供「編號、內容」，`start=1` 讓編號從1開始，對上輪數。所以「機率超過0.9就停」在這個例子立即錯。用較高門檻不能解決所有失準，需要另外做[機率校準](https://birdhackor.github.io/tiny-perceptron-vlm/9.9.html)與停止規則評估，記下每個規則答錯幾題、用了幾輪及多少時間。也可以學一個停止機率並在訓練中加入計算代價；ACT（Adaptive Computation Time，自適應計算時間）是早期研究之一。完整學習停止器還涉及離散動作或可微近似，本課列為進階，不要求學生先實作才能理解固定輪數。

第19章的 Auto 因而先做**開始前選預算**：先用另一個自己訓練的小模型**機率判斷器**，判斷題目屬於哪類、必要資訊是否齊全（[19.16](https://birdhackor.github.io/tiny-perceptron-vlm/19.16.html)），再依[決策資料](https://birdhackor.github.io/tiny-perceptron-vlm/9.14.html)（policy split，專門拿來選規則的一份資料）上選定的規則決定 Looped 解題器跑幾輪（[19.17](https://birdhackor.github.io/tiny-perceptron-vlm/19.17.html)）；它不假裝能讀到隱藏的正確答案。若改成 low 失敗再重跑 high，兩次工作都要記成本。若想從 low 跑完的狀態接著多跑幾輪，而不是從頭重算，模型與服務程式就得能把那份隱藏狀態存下來再接回去；第19章的 Looped 解題器每次都從原題目重新開始跑，沒有這種接續，改檔就是從頭重跑。

練習說明「先看過正確答案再選最好的一輪」有什麼用途。它可作固定評估紀錄上的回看上界，也就是[7.25](https://birdhackor.github.io/tiny-perceptron-vlm/7.25.html)的離線理想上界，提示尚有多少選擇空間；不代表一個能部署、免費且一定找得到好輪數的控制器。

來源：[ACT，2016](https://arxiv.org/abs/1603.08983)。
